# PV Forecasting (Restart from Original)
This notebook is a clean, consistent rebuild of the original with the **minimum set of fixes** needed to stop the “always sunny / too high bell curve” behavior.

Key fixes:
- **Proper forecasting dataset**: model sees **past PV + past weather** (plant state) and **future weather** (forecast drivers).
- **No leakage**: scalers fit only on training split (and per fold if you use CV).
- **Seq2seq model**: predicts a value **per horizon step**, not a whole day from a single timestep.
- **Train on log1p(PV)** to reduce peak overshoot; convert back for plotting.


In [1]:

# --- Settings ---
RUN_TRAINING = True

LOOKBACK = 168   # past hours (7 days)
HORIZON  = 24    # forecast hours
TEST_HOURS = 744 # holdout length in HOURS (not sequences)

LOCAL_TZ = None  # e.g. "Europe/Berlin" if your PV is local-time aligned. Leave None to use UTC.

SEED = 42


In [2]:
import os, glob, random, math, time, logging
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils import clip_grad_norm_, spectral_norm

from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error

from tqdm.auto import tqdm

# ---- Reproducibility ----
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device


device(type='cuda')

In [ ]:
# ---- Logging / experiment folder ----
RUN_DIR = Path("runs") / time.strftime("%Y%m%d-%H%M%S")
RUN_DIR.mkdir(parents=True, exist_ok=True)

LOG_PATH = RUN_DIR / "train.log"
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(message)s",
    handlers=[
        logging.StreamHandler(),
        logging.FileHandler(LOG_PATH, mode="w", encoding="utf-8"),
    ],
)
logger = logging.getLogger("pv_forecast")

logger.info(f"Device: {device}")
logger.info(f"Run dir: {RUN_DIR.resolve()}")


## Load & merge data

In [3]:

pv_data = pd.read_csv("../data/PV_2022_hourly.csv")
weather_data = pd.read_csv("../data/Weather_clean_NEMS.csv")

df_pv = pv_data.copy()
df_weather = weather_data.copy()

df_pv["TimestampInUtc"] = pd.to_datetime(df_pv["TimestampInUtc"], utc=True, errors="coerce")
df_weather["timestamp"] = pd.to_datetime(df_weather["timestamp"], utc=True, errors="coerce")

df_merged = df_pv.rename(columns={"TimestampInUtc":"ts"}).merge(
    df_weather.rename(columns={"timestamp":"ts"}),
    on="ts", how="inner"
).sort_values("ts").reset_index(drop=True)

# Optional: convert to local timezone BEFORE creating hour/month/day features
if LOCAL_TZ is not None:
    df_merged["ts_local"] = df_merged["ts"].dt.tz_convert(LOCAL_TZ)
else:
    df_merged["ts_local"] = df_merged["ts"]  # keep UTC

df_merged["hour"] = df_merged["ts_local"].dt.hour
df_merged["day_of_year"] = df_merged["ts_local"].dt.dayofyear
df_merged["month"] = df_merged["ts_local"].dt.month

df_merged["hour_sin"]  = np.sin(2*np.pi*df_merged["hour"]/24)
df_merged["hour_cos"]  = np.cos(2*np.pi*df_merged["hour"]/24)
df_merged["month_sin"] = np.sin(2*np.pi*df_merged["month"]/12)
df_merged["month_cos"] = np.cos(2*np.pi*df_merged["month"]/12)
df_merged["doy_sin"]   = np.sin(2*np.pi*df_merged["day_of_year"]/365.0)
df_merged["doy_cos"]   = np.cos(2*np.pi*df_merged["day_of_year"]/365.0)

# Clean PV (no negatives)
df_merged["pv"] = df_merged["pv"].clip(lower=0)

df_merged.head()


,ts,pv,Temperature,Sunshine Duration,Shortwave Radiation,Direct Shortwave Radiation,Diffuse Shortwave Radiation,Snowfall Amount,Relative Humidity,Cloud Cover Total,ts_local,hour,day_of_year,month,hour_sin,hour_cos,month_sin,month_cos,doy_sin,doy_cos
0,2022-01-01 00:00:00+00:00,0.0,9.657269,0.0,0.0,0.0,0.0,0.0,29.0,0.0,2022-01-01 00:00:00+00:00,0,1,1,0.000000,1.000000,0.5,0.866025,0.017213,0.999852
1,2022-01-01 01:00:00+00:00,0.0,9.457270,0.0,0.0,0.0,0.0,0.0,29.0,0.0,2022-01-01 01:00:00+00:00,1,1,1,0.258819,0.965926,0.5,0.866025,0.017213,0.999852
2,2022-01-01 02:00:00+00:00,0.0,9.787270,0.0,0.0,0.0,0.0,0.0,28.0,0.0,2022-01-01 02:00:00+00:00,2,1,1,0.500000,0.866025,0.5,0.866025,0.017213,0.999852
3,2022-01-01 03:00:00+00:00,0.0,9.467270,0.0,0.0,0.0,0.0,0.0,28.0,0.0,2022-01-01 03:00:00+00:00,3,1,1,0.707107,0.707107,0.5,0.866025,0.017213,0.999852
4,2022-01-01 04:00:00+00:00,0.0,8.797270,0.0,0.0,0.0,0.0,0.0,30.0,0.0,2022-01-01 04:00:00+00:00,4,1,1,0.866025,0.500000,0.5,0.866025,0.017213,0.999852


## Features
We keep the original weather + cyclic time features, and **add past PV** as an input (plant state). Future PV is unknown, so we provide a `pv_known` flag and set future pv input to 0.

Target is **log1p(pv)** to reduce overshoot.


In [4]:

TARGET_COL = "pv"
df_merged["y_log"] = np.log1p(df_merged[TARGET_COL].values.astype(np.float32))

CONTINUOUS_FEATURES = [
    "Temperature", "Sunshine Duration", "Shortwave Radiation",
    "Direct Shortwave Radiation", "Diffuse Shortwave Radiation",
    "Snowfall Amount", "Relative Humidity", "Cloud Cover Total"
]
# keep only those that exist (robust to column name changes)
CONTINUOUS_FEATURES = [c for c in CONTINUOUS_FEATURES if c in df_merged.columns]

CYCLIC_FEATURES = ["hour_sin","hour_cos","month_sin","month_cos","doy_sin","doy_cos"]

# Inputs we will build:
# - continuous weather (scaled)
# - cyclic time (not scaled)
# - pv_in (past pv; future pv_in=0)
# - pv_known flag (1 in history, 0 in future)


## Build forecasting tensors (history + future weather)
For each sample starting at time t:
- Input sequence length = LOOKBACK + HORIZON
  - first LOOKBACK steps: past weather/time + past pv
  - next HORIZON steps: future weather/time + pv_in=0 (unknown)
- Target = PV for next HORIZON hours (t .. t+HORIZON-1) in log space.


In [5]:

def build_forecast_tensors(df, lookback, horizon, continuous_cols, cyclic_cols, target_log_col="y_log"):
    Xc = df[continuous_cols].values.astype(np.float32)           # (N, C)
    Xcy = df[cyclic_cols].values.astype(np.float32)             # (N, K)
    pv = df["pv"].values.astype(np.float32)                     # (N,)
    ylog = df[target_log_col].values.astype(np.float32)         # (N,)

    N = len(df)
    seq_len = lookback + horizon
    X_list, Y_list = [], []

    # last start index so that we have lookback history and horizon future
    # Need history ending at t-1, and target from t..t+horizon-1
    for t in range(lookback, N - horizon):
        # history indices: t-lookback .. t-1 (length lookback)
        h_idx = slice(t - lookback, t)
        # future indices: t .. t+horizon-1 (length horizon)
        f_idx = slice(t, t + horizon)

        # Build per-step features for history and future
        # Weather/time available for both history and future
        hist_cont = Xc[h_idx]
        fut_cont  = Xc[f_idx]
        hist_cyc  = Xcy[h_idx]
        fut_cyc   = Xcy[f_idx]

        # PV input: known in history, unknown in future
        pv_hist = pv[h_idx].reshape(-1, 1)
        pv_fut  = np.zeros((horizon, 1), dtype=np.float32)

        known_hist = np.ones((lookback, 1), dtype=np.float32)
        known_fut  = np.zeros((horizon, 1), dtype=np.float32)

        X_hist = np.concatenate([hist_cont, hist_cyc, pv_hist, known_hist], axis=1)
        X_fut  = np.concatenate([fut_cont,  fut_cyc,  pv_fut,  known_fut],  axis=1)

        X_seq = np.concatenate([X_hist, X_fut], axis=0)  # (lookback+horizon, F)

        # Target: next horizon PV in log space (t..t+horizon-1)
        Y = ylog[f_idx]  # (horizon,)

        X_list.append(X_seq)
        Y_list.append(Y)

    X = np.stack(X_list, axis=0)  # (S, T, F)
    Y = np.stack(Y_list, axis=0)  # (S, horizon)
    return X, Y

X_raw, Y_log = build_forecast_tensors(df_merged, LOOKBACK, HORIZON, CONTINUOUS_FEATURES, CYCLIC_FEATURES, "y_log")
print("X_raw:", X_raw.shape, "Y_log:", Y_log.shape)


X_raw: (8566, 192, 16) Y_log: (8566, 24)


## Train/holdout split
We split by **time**: the last `TEST_HOURS` hours become the holdout window.

Since each training sample needs LOOKBACK history and HORIZON future, we convert hours to sequences safely.


In [6]:

# Convert TEST_HOURS to approx sequences in X_raw space.
# Each sequence predicts a horizon starting at time t. The last sequence start corresponds to near the end of df.
# We'll take the last `TEST_HOURS` *starts* as test sequences (reasonable for day-ahead evaluation).
TEST_SIZE_SEQ = TEST_HOURS

X_train_raw = X_raw[:-TEST_SIZE_SEQ]
Y_train_log = Y_log[:-TEST_SIZE_SEQ]
X_test_raw  = X_raw[-TEST_SIZE_SEQ:]
Y_test_log  = Y_log[-TEST_SIZE_SEQ:]

print("Train:", X_train_raw.shape, Y_train_log.shape)
print("Test :", X_test_raw.shape,  Y_test_log.shape)


Train: (7822, 192, 16) (7822, 24)
Test : (744, 192, 16) (744, 24)


## Scaling (train-only)
Scale only continuous weather columns. Cyclic + pv_in + pv_known stay unscaled.


In [7]:

# feature layout: [cont(C), cyc(6), pv_in(1), pv_known(1)]
C = len(CONTINUOUS_FEATURES)
K = len(CYCLIC_FEATURES)
F_total = X_train_raw.shape[-1]
assert F_total == C + K + 2

scaler_X = StandardScaler()

def scale_X(train_X, test_X):
    tr = train_X.copy()
    te = test_X.copy()
    tr_cont = tr[:, :, :C].reshape(-1, C)
    te_cont = te[:, :, :C].reshape(-1, C)
    scaler_X.fit(tr_cont)
    tr[:, :, :C] = scaler_X.transform(tr_cont).reshape(tr.shape[0], tr.shape[1], C)
    te[:, :, :C] = scaler_X.transform(te_cont).reshape(te.shape[0], te.shape[1], C)
    return tr.astype(np.float32), te.astype(np.float32)

X_train, X_test = scale_X(X_train_raw, X_test_raw)


## Dataset / DataLoader

In [8]:

class SeqDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.float32)
    def __len__(self): return self.X.shape[0]
    def __getitem__(self, idx): return self.X[idx], self.y[idx]

train_loader = DataLoader(SeqDataset(X_train, Y_train_log), batch_size=32, shuffle=True, num_workers=0)
test_loader  = DataLoader(SeqDataset(X_test,  Y_test_log),  batch_size=64, shuffle=False, num_workers=0)


## TCN seq2seq model (replicate padding)

In [9]:

class TemporalBlock(nn.Module):
    def __init__(self, in_channels, out_channels, kernel_size, dilation, dropout):
        super().__init__()
        pad = (kernel_size - 1) * dilation

        self.pad1  = nn.ReplicationPad1d((pad, 0))
        self.conv1 = spectral_norm(nn.Conv1d(in_channels, out_channels, kernel_size,
                                             stride=1, padding=0, dilation=dilation))
        self.relu1 = nn.ReLU()
        self.drop1 = nn.Dropout(dropout)

        self.pad2  = nn.ReplicationPad1d((pad, 0))
        self.conv2 = spectral_norm(nn.Conv1d(out_channels, out_channels, kernel_size,
                                             stride=1, padding=0, dilation=dilation))
        self.relu2 = nn.ReLU()
        self.drop2 = nn.Dropout(dropout)

        self.down = nn.Conv1d(in_channels, out_channels, 1) if in_channels != out_channels else None
        self.out_relu = nn.ReLU()

    def forward(self, x):
        out = self.drop1(self.relu1(self.conv1(self.pad1(x))))
        out = self.drop2(self.relu2(self.conv2(self.pad2(out))))
        res = x if self.down is None else self.down(x)
        return self.out_relu(out + res)

class TCNForecaster(nn.Module):
    def __init__(self, input_size, num_channels=(64,64,64), kernel_size=3, dropout=0.1, horizon=24):
        super().__init__()
        layers = []
        for i, ch in enumerate(num_channels):
            in_ch = input_size if i == 0 else num_channels[i-1]
            dilation = 2**i
            layers.append(TemporalBlock(in_ch, ch, kernel_size, dilation, dropout))
        self.tcn = nn.Sequential(*layers)
        self.head = nn.Linear(num_channels[-1], 1)
        self.horizon = horizon

    def forward(self, x):
        # x: (B, T, F)
        x_ch = x.transpose(1, 2)         # (B, F, T)
        h = self.tcn(x_ch)               # (B, C, T)
        h = h.transpose(1, 2)            # (B, T, C)
        y = self.head(h).squeeze(-1)     # (B, T)

        # We only care about the last HORIZON steps (future window)
        y = y[:, -self.horizon:]         # (B, HORIZON)
        return y


## Train / eval (log space)

In [ ]:
criterion = torch.nn.L1Loss()

def metrics_pv_units(P_log, Y_log):
    """Compute scalar metrics in original PV units (assumes y_log = log1p(pv))."""
    P = np.expm1(np.clip(P_log, a_min=0, a_max=None))
    Y = np.expm1(np.clip(Y_log, a_min=0, a_max=None))
    mse = mean_squared_error(Y.reshape(-1), P.reshape(-1))
    rmse = math.sqrt(mse)
    mae = mean_absolute_error(Y.reshape(-1), P.reshape(-1))
    return mse, rmse, mae

def train_one_epoch(model, loader, optimizer, *, grad_clip=1.0, pbar=True):
    model.train()
    total_loss = 0.0
    total_grad = 0.0
    n = 0

    it = loader
    if pbar:
        it = tqdm(loader, desc="train", leave=False)

    for X, Y in it:
        X, Y = X.to(device), Y.to(device)

        optimizer.zero_grad(set_to_none=True)
        pred = model(X)
        loss = criterion(pred, Y)
        loss.backward()

        if grad_clip is not None:
            grad_norm = float(clip_grad_norm_(model.parameters(), grad_clip))
        else:
            grad_norm = float("nan")

        optimizer.step()

        bs = X.size(0)
        total_loss += float(loss.item()) * bs
        total_grad += grad_norm * bs
        n += bs

        if pbar:
            it.set_postfix(loss=float(loss.item()), grad=grad_norm)

    return {
        "loss": total_loss / max(n, 1),
        "grad_norm": total_grad / max(n, 1),
    }

@torch.no_grad()
def eval_on_loader(model, loader):
    model.eval()
    total_loss = 0.0
    n = 0
    preds, trues = [], []
    for Xb, yb in loader:
        Xb = Xb.to(device)
        yb = yb.to(device)
        pred = model(Xb)
        loss = criterion(pred, yb)
        bs = Xb.size(0)
        total_loss += float(loss.item()) * bs
        n += bs
        preds.append(pred.detach().cpu().numpy())
        trues.append(yb.detach().cpu().numpy())
    P = np.concatenate(preds, axis=0) if preds else np.empty((0, HORIZON), dtype=np.float32)
    Y = np.concatenate(trues, axis=0) if trues else np.empty((0, HORIZON), dtype=np.float32)
    return total_loss / max(n, 1), P, Y

@torch.no_grad()
def eval_daily_from_samples(model, X_samples, Y_samples, *, step=24):
    """Evaluate one forecast per day by taking every `step`-th sample in the test set."""
    model.eval()

    idxs = np.arange(0, len(X_samples), step)
    if len(idxs) == 0:
        return float("nan"), np.empty((0, HORIZON), dtype=np.float32), np.empty((0, HORIZON), dtype=np.float32)

    Xb = torch.tensor(X_samples[idxs], dtype=torch.float32, device=device)
    yb = torch.tensor(Y_samples[idxs], dtype=torch.float32, device=device)

    pred = model(Xb)
    loss = criterion(pred, yb)

    return float(loss.item()), pred.detach().cpu().numpy(), yb.detach().cpu().numpy()


## Run training

In [ ]:
EPOCHS = 20
LR = 1e-3

PATIENCE = 5          # stop if no MAE improvement for this many epochs
MIN_DELTA = 0.0       # require at least this improvement to reset patience

model = TCNForecaster(
    input_size=X_train.shape[-1],
    num_channels=(64,64,64,64),
    kernel_size=3,
    dropout=0.1,
    horizon=HORIZON
).to(device)

optimizer = torch.optim.Adam(model.parameters(), lr=LR)

# Optional TensorBoard (works if tensorboard is installed)
try:
    from torch.utils.tensorboard import SummaryWriter
    writer = SummaryWriter(log_dir=str(RUN_DIR / "tb"))
    logger.info("TensorBoard: enabled")
except Exception as e:
    writer = None
    logger.info(f"TensorBoard: disabled ({e})")

history = []
best_mae = float("inf")
best_epoch = -1
epochs_no_improve = 0

logger.info("Starting training ...")
logger.info(f"Train samples: {len(train_loader.dataset)} | Test samples: {len(test_loader.dataset)}")

for epoch in range(1, EPOCHS + 1):
    t0 = time.time()

    tr = train_one_epoch(model, train_loader, optimizer, grad_clip=1.0, pbar=True)
    te_loss, P_log_all, Y_log_all = eval_on_loader(model, test_loader)

    # Daily evaluation (one forecast per day: every 24th hour-start sample)
    daily_loss, P_log_day, Y_log_day = eval_daily_from_samples(model, X_test, Y_test_log, step=24)

    # Metrics in PV units (scalar) from daily forecasts
    mse, rmse, mae = metrics_pv_units(P_log_day, Y_log_day)

    # Early stopping based on MAE (PV units)
    improved = mae < (best_mae - MIN_DELTA)
    if improved:
        best_mae = mae
        best_epoch = epoch
        epochs_no_improve = 0

        torch.save(
            {
                "epoch": epoch,
                "model_state": model.state_dict(),
                "optimizer_state": optimizer.state_dict(),
                "mae": mae,
                "rmse": rmse,
                "mse": mse,
            },
            RUN_DIR / "best_model.pt",
        )
    else:
        epochs_no_improve += 1

    row = {
        "epoch": epoch,
        "train_loss_log": tr["loss"],
        "train_grad_norm": tr["grad_norm"],
        "test_loss_log": te_loss,
        "daily_loss_log": daily_loss,
        "daily_mse_pv": mse,
        "daily_rmse_pv": rmse,
        "daily_mae_pv": mae,
        "lr": optimizer.param_groups[0]["lr"],
        "best_mae_pv": best_mae,
        "best_epoch": best_epoch,
        "no_improve": epochs_no_improve,
        "sec": time.time() - t0,
    }
    history.append(row)

    msg = (
        f"Epoch {epoch:03d} | "
        f"train_log={row['train_loss_log']:.4f} | "
        f"test_log={row['test_loss_log']:.4f} | "
        f"daily_log={row['daily_loss_log']:.4f} | "
        f"MAE={row['daily_mae_pv']:.3f} | "
        f"RMSE={row['daily_rmse_pv']:.3f} | "
        f"best_MAE={row['best_mae_pv']:.3f} (ep {best_epoch}) | "
        f"no_improve={epochs_no_improve}/{PATIENCE}"
    )
    logger.info(msg)

    if writer is not None:
        writer.add_scalar("loss/train_log", row["train_loss_log"], epoch)
        writer.add_scalar("loss/test_log", row["test_loss_log"], epoch)
        writer.add_scalar("loss/daily_log", row["daily_loss_log"], epoch)
        writer.add_scalar("grad/train_grad_norm", row["train_grad_norm"], epoch)
        writer.add_scalar("pv/daily_mae", row["daily_mae_pv"], epoch)
        writer.add_scalar("pv/daily_rmse", row["daily_rmse_pv"], epoch)
        writer.add_scalar("pv/daily_mse", row["daily_mse_pv"], epoch)
        writer.add_scalar("lr", row["lr"], epoch)

    # Save metrics each epoch
    pd.DataFrame(history).to_csv(RUN_DIR / "metrics.csv", index=False)

    # ---- EARLY STOP ----
    if epochs_no_improve >= PATIENCE:
        logger.info(f"Early stopping at epoch {epoch} (best MAE {best_mae:.3f} at epoch {best_epoch})")
        break

# ---- load best checkpoint at the end ----
ckpt_path = RUN_DIR / "best_model.pt"
ckpt = torch.load(ckpt_path, map_location=device)
model.load_state_dict(ckpt["model_state"])
logger.info(f"Loaded best model from epoch {ckpt['epoch']} | best MAE {ckpt['mae']:.3f} | checkpoint: {ckpt_path}")

if writer is not None:
    writer.flush()
    writer.close()

metrics_df = pd.DataFrame(history)
metrics_df


Epoch 001 | train loss 0.24906 | test loss 0.16575 | PV RMSE 9.169 | PV MAE 3.640
Epoch 005 | train loss 0.07218 | test loss 0.22482 | PV RMSE 10.506 | PV MAE 4.658
Epoch 010 | train loss 0.05119 | test loss 0.24092 | PV RMSE 11.853 | PV MAE 5.130
Epoch 015 | train loss 0.03683 | test loss 0.20809 | PV RMSE 11.209 | PV MAE 4.664
Epoch 020 | train loss 0.02748 | test loss 0.22656 | PV RMSE 11.736 | PV MAE 5.028


## Plot an example 24h forecast (holdout)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# ---- Re-run daily evaluation with the FINAL (best-loaded) model ----
daily_loss, P_log_day, Y_log_day = eval_daily_from_samples(model, X_test, Y_test_log, step=24)

P_day = np.expm1(np.clip(P_log_day, a_min=0, a_max=None))
Y_day = np.expm1(np.clip(Y_log_day, a_min=0, a_max=None))

mse_d = np.mean((P_day - Y_day) ** 2, axis=1)
rmse_d = np.sqrt(mse_d)
mae_d = np.mean(np.abs(P_day - Y_day), axis=1)

print(f"Daily eval (log-space MAE): {daily_loss:.4f}")
print(f"Daily eval (PV units): MAE={mae_d.mean():.3f} | RMSE={rmse_d.mean():.3f}")

# ---- Plot training curves (if available) ----
if 'metrics_df' in globals() and len(metrics_df) > 0:
    plt.figure(figsize=(7,4))
    plt.plot(metrics_df["epoch"], metrics_df["train_loss_log"], label="train log MAE")
    plt.plot(metrics_df["epoch"], metrics_df["test_loss_log"], label="test log MAE")
    plt.plot(metrics_df["epoch"], metrics_df["daily_loss_log"], label="daily log MAE")
    plt.xlabel("Epoch")
    plt.ylabel("MAE (log space)")
    plt.title("Loss curves")
    plt.legend()
    plt.grid(True)
    plt.show()

    plt.figure(figsize=(7,4))
    plt.plot(metrics_df["epoch"], metrics_df["daily_mae_pv"], label="daily MAE (PV)")
    plt.plot(metrics_df["epoch"], metrics_df["daily_rmse_pv"], label="daily RMSE (PV)")
    plt.xlabel("Epoch")
    plt.ylabel("Error (PV units)")
    plt.title("Daily metrics (PV units)")
    plt.legend()
    plt.grid(True)
    plt.show()

# ---- Plot example days ----
def plot_days(P, Y, rmse_d, mae_d, K=9, mode="random", seed=42):
    N = len(P)
    if N == 0:
        print("No days to plot.")
        return

    if mode == "random":
        rng = np.random.default_rng(seed)
        idxs = rng.choice(N, size=min(K, N), replace=False)
    elif mode == "worst":
        idxs = np.argsort(rmse_d)[-min(K, N):][::-1]
    elif mode == "best":
        idxs = np.argsort(rmse_d)[:min(K, N)]
    else:
        raise ValueError("mode must be 'random', 'worst', or 'best'")

    cols = int(np.ceil(np.sqrt(len(idxs))))
    rows = int(np.ceil(len(idxs) / cols))

    plt.figure(figsize=(cols*4, rows*3))
    for i, idx in enumerate(idxs, 1):
        plt.subplot(rows, cols, i)
        plt.plot(Y[idx], label="true")
        plt.plot(P[idx], label="pred")
        plt.title(f"day={idx} | RMSE={rmse_d[idx]:.2f} | MAE={mae_d[idx]:.2f}")
        plt.xlabel("Hour ahead")
        plt.ylabel("PV")
        if i == 1:
            plt.legend()
    plt.tight_layout()
    plt.show()

plot_days(P_day, Y_day, rmse_d, mae_d, K=6, mode="random")
plot_days(P_day, Y_day, rmse_d, mae_d, K=6, mode="worst")


NameError: name 'P' is not defined